---

# Часть 2.5: Pipeline в sklearn

Перед тем как работать с реальными данными, познакомимся с важным инструментом - **Pipeline**

Примеры использования Pipeline в scikit-learn
--------------------------------------------------
- базовой работы с Pipeline;
- сложных пайплайнов с feature engineering;
- сохранения/загрузки модели;
- сравнения с ручным подходом.

## 📖 Что такое Pipeline?

**Pipeline** - это последовательность шагов обработки данных и обучения модели, объединенных в один объект.

### Зачем нужен Pipeline?

1. **Избегает утечки данных (data leakage)**
   - Все трансформации применяются правильно на train/test
   - fit() выполняется только на train данных

2. **Упрощает код**
   - Вместо множества отдельных шагов - один объект
   - Легко воспроизвести всю цепочку обработки

3. **Удобен для production**
   - Сохраняется весь процесс обработки
   - Легко применить к новым данным

4. **Интеграция с grid search**
   - Можно оптимизировать параметры всех шагов сразу

## ❌ Что было бы БЕЗ Pipeline?

In [8]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    StandardScaler,
    PolynomialFeatures,
    OneHotEncoder
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import joblib
import warnings
warnings.filterwarnings('ignore')

# 1. Создаём синтетические данные для примера
np.random.seed(42)
n_samples = 1000

X = pd.DataFrame({
    'age': np.random.normal(35, 10, n_samples),
    'income': np.random.normal(50000, 15000, n_samples),
    'score': np.random.uniform(300, 850, n_samples),
    'category': np.random.choice(['A', 'B', 'C'], n_samples)
})

# Целевая переменная (бинарная классификация)
y = (X['income'] > 45000).astype(int)

# Разбиваем на train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("🚀 Pipeline Examples\n")
print("=" * 60)

🚀 Pipeline Examples



In [11]:
scaler = StandardScaler()
scaler.fit(X_train[['age', 'income', 'score']])  # fit только на train

X_train_scaled = scaler.transform(X_train[['age', 'income', 'score']])
X_test_scaled = scaler.transform(X_test[['age', 'income', 'score']])  # transform, не fit!


model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

y_pred_manual = model.predict(X_test_scaled)
acc_manual = accuracy_score(y_test, y_pred_manual)

print(f"Accuracy: {accuracy_score(y_test, y_pred_manual):.4f}")
print("\n⚠️ Проблемы:")
print("   • Много шагов вручную")
print("   • Легко ошибиться (забыть transform или сделать fit на test)")
print("   • Сложно сохранить и воспроизвести")
print("   • Риск data leakage")

Accuracy: 0.9900

⚠️ Проблемы:
   • Много шагов вручную
   • Легко ошибиться (забыть transform или сделать fit на test)
   • Сложно сохранить и воспроизвести
   • Риск data leakage


## ✅ Что ДАЕТ Pipeline?

In [12]:
print("С PIPELINE - просто и безопасно:")
print("="*50)

# Создаем Pipeline - последовательность шагов
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(random_state=42))
])

pipeline.fit(X_train[['age', 'income', 'score']], y_train)
y_pred_pipe = pipeline.predict(X_test[['age', 'income', 'score']])
acc_pipe = accuracy_score(y_test, y_pred_pipe)

print(f"Accuracy (Pipeline): {acc_pipe:.4f}")
print("✅ Преимущества:")
print("  • Всё в одном объекте")
print("  • Нет риска data leakage\n")

С PIPELINE - просто и безопасно:
Accuracy (Pipeline): 0.9900
✅ Преимущества:
  • Всё в одном объекте
  • Нет риска data leakage



## 🔧 Как работает Pipeline?

### При вызове fit():

```python
pipeline.fit(X_train, y_train)
```

Pipeline выполняет:
1. `scaler.fit(X_train)` - обучает scaler на train данных
2. `X_train_scaled = scaler.transform(X_train)` - трансформирует train
3. `model.fit(X_train_scaled, y_train)` - обучает модель на scaled данных

### При вызове predict():

```python
pipeline.predict(X_test)
```

Pipeline выполняет:
1. `X_test_scaled = scaler.transform(X_test)` - трансформирует test (БЕЗ fit!)
2. `predictions = model.predict(X_test_scaled)` - делает предсказания

**Важно:** На test данных НЕ вызывается fit() - используются параметры с train!

## Pipeline с категориальными признаками

In [18]:
print("3. Pipeline с категориальными признаками:")
print("=" * 50)

# Определим трансформеры для разных типов данных
numeric_features = ['age', 'income', 'score']
categorical_features = ['category']

numeric_transformer = Pipeline([
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Полный пайплайн
pipeline_cat = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state=42))
])

pipeline_cat.fit(X_train, y_train)
y_pred_cat = pipeline_cat.predict(X_test)
acc_cat = accuracy_score(y_test, y_pred_cat)

print(f"Accuracy (с категориальными): {acc_cat:.4f}")
print("💡 Pipeline автоматически обрабатывает все типы данных\n")

3. Pipeline с категориальными признаками:
Accuracy (с категориальными): 1.0000
💡 Pipeline автоматически обрабатывает все типы данных



## 🎯 Pipeline с feature engineering

In [20]:
from sklearn.preprocessing import PolynomialFeatures

print("Pipeline с полиномиальными признаками:")
print("-" * 50)

pipeline_poly = Pipeline([
    ('scaler', StandardScaler()), # Шаг 1: нормализация
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),# Шаг 2: полиномиальные признаки
    ('model', LogisticRegression(random_state=42, max_iter=1000)) # Шаг 3: модель
])

pipeline_poly.fit(X_train[['age', 'income', 'score']], y_train)
y_pred_poly = pipeline_poly.predict(X_test[['age', 'income', 'score']])
acc_poly = accuracy_score(y_test, y_pred_poly)

print(f"Accuracy (полиномиальные признаки): {acc_poly:.4f}")
print("🔍 Pipeline добавил x², x*y и другие комбинации\n")

Pipeline с полиномиальными признаками:
--------------------------------------------------
Accuracy (полиномиальные признаки): 0.9950
🔍 Pipeline добавил x², x*y и другие комбинации



## 💾 Сохранение и загрузка Pipeline

In [23]:
import joblib

print("Сохранение и загрузка Pipeline:")
print("-" * 50)

# Сохраняем
joblib.dump(pipeline_cat, 'pipeline_example.pkl')
print("✅ Pipeline сохранён в 'pipeline_example.pkl'")

# Загружаем
loaded_pipeline = joblib.load('pipeline_example.pkl')
print("✅ Pipeline загружен из файла")

# Проверяем
y_pred_loaded = loaded_pipeline.predict(X_test)
acc_loaded = accuracy_score(y_test, y_pred_loaded)

print(f"Accuracy (загруженный Pipeline): {acc_loaded:.4f}")
print("💡 Сохранены все параметры и шаги обработки\n")

Сохранение и загрузка Pipeline:
--------------------------------------------------
✅ Pipeline сохранён в 'pipeline_example.pkl'
✅ Pipeline загружен из файла
Accuracy (загруженный Pipeline): 1.0000
💡 Сохранены все параметры и шаги обработки



## 📝 Выводы по Pipeline

### Когда использовать Pipeline?

✅ **ВСЕГДА** когда есть preprocessing!

### Основные преимущества:

1. **Безопасность** - нет риска data leakage
2. **Простота** - чистый и понятный код
3. **Production-ready** - легко сохранить и использовать
4. **Воспроизводимость** - весь процесс в одном объекте

### В следующей части:

Будем использовать Pipeline для всех моделей на реальных торговых данных! 🚀